Copyright (c) MONAI Consortium  
Licensed under the Apache License, Version 2.0 (the "License");  
you may not use this file except in compliance with the License.  
You may obtain a copy of the License at  
&nbsp;&nbsp;&nbsp;&nbsp;http://www.apache.org/licenses/LICENSE-2.0  
Unless required by applicable law or agreed to in writing, software  
distributed under the License is distributed on an "AS IS" BASIS,  
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.  
See the License for the specific language governing permissions and  
limitations under the License.

# MONAI 101 tutorial

In this tutorial, we will introduce how simple it can be to run an end-to-end classification pipeline with MONAI.

These steps will be included in this tutorial, and each of them will take only a few lines of code:
- Dataset download
- Data pre-processing
- Define a DenseNet-121 and run training
- Check the results on test dataset

This tutorial will use about 7GB of GPU memory and 10 minutes to run.

## Setup imports

In [ ]:
import os
import shutil
import tempfile
import logging
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bioMONAI.data import *
from bioMONAI.core import *
from bioMONAI.metrics import ROCAUCMetric
from bioMONAI.datasets import download_file
from bioMONAI.core import parent_label, Path, accuracy
from bioMONAI.io import image_reader
from bioMONAI.losses import CrossEntropyLossFlat
from bioMONAI.data import CategoryBlock
from bioMONAI.callbacks import SaveModelCallback
from bioMONAI.visualize import plot_metrics
from bioMONAI.transforms import (
    ScaleIntensity,
    RandFlip,
    RandRotate,
    RandZoom,
)

from monai.config import print_config
from monai.networks.nets import DenseNet121
from monai.utils import set_determinism

from monai.transforms import LoadImageD, EnsureChannelFirstD, ScaleIntensityD, Compose
from monai.apps import MedNISTDataset

print_config()

## Setup data directory

You can specify a directory.  
This allows you to save results and reuse downloads.  
If not specified a temporary directory will be used.

In [ ]:
base_directory = '../../_data/'
if base_directory is not None:
    os.makedirs(base_directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if base_directory is None else base_directory
print(root_dir)

## Use MONAI transforms to preprocess data

Medical images require specialized methods for I/O, preprocessing, and augmentation.
They often follow specific formats, are handled with specific protocols, and the data arrays are often high-dimensional.

In this example, we will perform image loading, data format verification, and intensity scaling with three `monai.transforms` listed below, and compose a pipeline ready to be used in next steps.

## Prepare datasets using MONAI Apps

We use `MedNISTDataset` in MONAI Apps to download a dataset to the specified directory and perform the pre-processing steps in the `monai.transforms` compose.

The MedNIST dataset was gathered from several sets from [TCIA](https://wiki.cancerimagingarchive.net/display/Public/Data+Usage+Policies+and+Restrictions),
[the RSNA Bone Age Challenge](http://rsnachallenges.cloudapp.net/competitions/4),
and [the NIH Chest X-ray dataset](https://cloud.google.com/healthcare/docs/resources/public-datasets/nih-chest).

The dataset is kindly made available by [Dr. Bradley J. Erickson M.D., Ph.D.](https://www.mayo.edu/research/labs/radiology-informatics/overview) (Department of Radiology, Mayo Clinic)
under the Creative Commons [CC BY-SA 4.0 license](https://creativecommons.org/licenses/by-sa/4.0/).

If you use the MedNIST dataset, please acknowledge the source. 

In [ ]:
transform = Compose(
    [
        LoadImageD(keys="image", image_only=True),
        EnsureChannelFirstD(keys="image"),
        ScaleIntensityD(keys="image"),
    ]
)

In [ ]:
train_df = pd.DataFrame(MedNISTDataset(root_dir=root_dir, transform=transform, section="training", download=True).data)
val_df = pd.DataFrame(MedNISTDataset(root_dir=root_dir, transform=transform, section="validation", download=False, runtime_cache=True).data)
test_df = pd.DataFrame(MedNISTDataset(root_dir=root_dir, transform=transform, section="test", download=False, runtime_cache=True).data)
# Concatenate train and val dataframes
full_train_df = pd.concat([train_df.assign(is_valid=0), val_df.assign(is_valid=1)], ignore_index=True)

In [ ]:
data_ops = {
    'fn_col': ['image'],
    'label_col': ['class_name'],
    'valid_col': ['is_valid'],
    'seed': 0, 
    'bs': 512,
    'item_tfms': [],   # item transformations
    'shuffle': True,
}

data = BioDataLoaders.class_from_df(full_train_df, **data_ops)

test_data = test_biodataloader(data, test_df) # type:ignore

# print length of training, validation, and test datasets
print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items), '\ntest images:', len(test_data.items))

## Define a network and a supervised trainer

To train a model that can perform the classification task, we will use the DenseNet-121 which is known for its performance on the ImageNet dataset.

For a typical supervised training workflow, MONAI provides `SupervisedTrainer` to define the hyper-parameters.

In [ ]:
max_epochs = 5
device = get_device()
model = DenseNet121(spatial_dims=2, in_channels=1, out_channels=6)
loss_function = CrossEntropyLossFlat()
metrics = [accuracy]

trainer = fastTrainer(data, model, loss_fn=loss_function, metrics=metrics, show_summary=False, lr=1e-5)

## Run the training

In [ ]:
trainer.fit(max_epochs)

## Check the prediction on the test dataset

In [ ]:
evaluate_classification_model(trainer, test_data, metrics=accuracy, show_graph=True);